In [36]:
import pandas as pd
import zipfile

# Unzip the uploaded spam.csv.zip file
with zipfile.ZipFile('spam.csv.zip', 'r') as zip_ref:
    zip_ref.extractall('.')

# Load the extracted CSV file
df = pd.read_csv('spam.csv', encoding='latin-1')

# Keep relevant columns and clean column names
df = df[['v1', 'v2']]
df.columns = ['label', 'message']

# Drop duplicate rows to clean dataset
df = df.drop_duplicates(keep='first')

# Display basic information and shape
print("Dataset Shape:", df.shape)
print("\nClass Counts:\n", df['label'].value_counts())
print("\nFirst 5 Rows:")
print(df.head())

Dataset Shape: (5169, 2)

Class Counts:
 label
ham     4516
spam     653
Name: count, dtype: int64

First 5 Rows:
  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


In [37]:
# Map text labels to numerical values (ham: 0, spam: 1)
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

# Verify mapping by printing class distribution
print("Mapped Class Counts:")
print(df['label_num'].value_counts())

# Display converted columns
print("\nUpdated DataFrame Preview:")
print(df[['label', 'label_num', 'message']].head())

Mapped Class Counts:
label_num
0    4516
1     653
Name: count, dtype: int64

Updated DataFrame Preview:
  label  label_num                                            message
0   ham          0  Go until jurong point, crazy.. Available only ...
1   ham          0                      Ok lar... Joking wif u oni...
2  spam          1  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham          0  U dun say so early hor... U c already then say...
4   ham          0  Nah I don't think he goes to usf, he lives aro...


In [38]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Download required NLTK corpus
nltk.download('stopwords')

# Initialize stemmer and stop words set
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # Convert text to lowercase
    text = text.lower()

    # Remove special characters and punctuation (keep only alphabets and spaces)
    text = re.sub(r'[^a-z\s]', '', text)

    # Split text into individual tokens
    words = text.split()

    # Filter out stopwords and perform stemming
    cleaned_words = [stemmer.stem(word) for word in words if word not in stop_words]

    # Rejoin words into a cleaned sentence
    return ' '.join(cleaned_words)

# Apply preprocessing to all SMS messages
df['cleaned_message'] = df['message'].apply(preprocess_text)

# Display raw text vs cleaned text
print("Text Preprocessing Preview:")
print(df[['message', 'cleaned_message']].head())

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Text Preprocessing Preview:
                                             message  \
0  Go until jurong point, crazy.. Available only ...   
1                      Ok lar... Joking wif u oni...   
2  Free entry in 2 a wkly comp to win FA Cup fina...   
3  U dun say so early hor... U c already then say...   
4  Nah I don't think he goes to usf, he lives aro...   

                                     cleaned_message  
0  go jurong point crazi avail bugi n great world...  
1                              ok lar joke wif u oni  
2  free entri wkli comp win fa cup final tkt st m...  
3                u dun say earli hor u c alreadi say  
4          nah dont think goe usf live around though  


In [39]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF Vectorizer with top 3000 frequent features
tfidf = TfidfVectorizer(max_features=3000)

# Transform cleaned text into matrix of TF-IDF features
X = tfidf.fit_transform(df['cleaned_message']).toarray()
y = df['label_num'].values

# Display shape of feature matrix
print("Feature Matrix (X) Shape:", X.shape)
print("Target Vector (y) Shape:", y.shape)

Feature Matrix (X) Shape: (5169, 3000)
Target Vector (y) Shape: (5169,)


In [40]:
from sklearn.model_selection import train_test_split

# Split dataset into 80% training and 20% testing sets using stratified sampling
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Display shapes of split datasets
print("Training Set Shape (X_train):", X_train.shape)
print("Testing Set Shape (X_test):", X_test.shape)
print("Training Target Shape (y_train):", y_train.shape)
print("Testing Target Shape (y_test):", y_test.shape)

Training Set Shape (X_train): (4135, 3000)
Testing Set Shape (X_test): (1034, 3000)
Training Target Shape (y_train): (4135,)
Testing Target Shape (y_test): (1034,)


In [41]:
from sklearn.naive_bayes import MultinomialNB

# Initialize the Multinomial Naive Bayes model
model = MultinomialNB()

# Train the model using training features and labels
model.fit(X_train, y_train)

print("Model training completed successfully!")

Model training completed successfully!


In [42]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Predict target labels for unseen test features
y_pred = model.predict(X_test)

# Calculate and display performance metrics
print("--- Model Evaluation Metrics ---")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Ham (0)', 'Spam (1)']))

--- Model Evaluation Metrics ---
Accuracy Score: 97.29%

Confusion Matrix:
[[902   1]
 [ 27 104]]

Classification Report:
              precision    recall  f1-score   support

     Ham (0)       0.97      1.00      0.98       903
    Spam (1)       0.99      0.79      0.88       131

    accuracy                           0.97      1034
   macro avg       0.98      0.90      0.93      1034
weighted avg       0.97      0.97      0.97      1034



In [43]:
def predict_sms(custom_message):
    # Preprocess custom message
    cleaned = preprocess_text(custom_message)

    # Vectorize cleaned text using trained TF-IDF instance
    vectorized = tfidf.transform([cleaned]).toarray()

    # Predict label and confidence score
    prediction = model.predict(vectorized)[0]
    probabilities = model.predict_proba(vectorized)[0]

    label_result = "SPAM" if prediction == 1 else "HAM"
    confidence_score = probabilities[prediction] * 100

    return label_result, confidence_score

# Test sample messages
sample_spam = "Urgent! You have won $1000 cash prize. Call now to claim."
sample_ham = "Hey, are we still meeting for lunch today?"

result_spam, conf_spam = predict_sms(sample_spam)
result_ham, conf_ham = predict_sms(sample_ham)

print("--- Custom Input Inference Results ---")
print(f"Message: '{sample_spam}'")
print(f"Prediction: {result_spam} (Confidence: {conf_spam:.2f}%)\n")

print(f"Message: '{sample_ham}'")
print(f"Prediction: {result_ham} (Confidence: {conf_ham:.2f}%)")

--- Custom Input Inference Results ---
Message: 'Urgent! You have won $1000 cash prize. Call now to claim.'
Prediction: SPAM (Confidence: 98.79%)

Message: 'Hey, are we still meeting for lunch today?'
Prediction: HAM (Confidence: 99.64%)


In [44]:
import joblib

# Save model and vectorizer
joblib.dump(model, 'spam_model.pkl')
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')

print("Model and vectorizer saved successfully!")

Model and vectorizer saved successfully!


In [45]:
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report

# Load public social media spam comments dataset directly via GitHub mirror
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
df_social = pd.read_csv(url, sep='\t', header=None, names=['label', 'message'])

# Encode target labels
df_social['label_num'] = df_social['label'].map({'ham': 0, 'spam': 1})

# Preprocess comments using your existing pipeline
print("Preprocessing social media comments...")
df_social['cleaned_message'] = df_social['message'].apply(preprocess_text)

# Vectorize cleaned text using your trained TF-IDF instance
X_social = tfidf.transform(df_social['cleaned_message']).toarray()
y_social = df_social['label_num']

# Predict using trained model
y_social_pred = model.predict(X_social)

# Display Evaluation Results
print("--- Social Media Dataset Accuracy Evaluation ---")
print(f"Overall Accuracy: {accuracy_score(y_social, y_social_pred) * 100:.2f}%\n")
print("Detailed Performance Matrix:")
print(classification_report(y_social, y_social_pred, target_names=['Ham / Clean', 'Spam']))

Preprocessing social media comments...
--- Social Media Dataset Accuracy Evaluation ---
Overall Accuracy: 97.72%

Detailed Performance Matrix:
              precision    recall  f1-score   support

 Ham / Clean       0.97      1.00      0.99      4825
        Spam       1.00      0.83      0.91       747

    accuracy                           0.98      5572
   macro avg       0.99      0.92      0.95      5572
weighted avg       0.98      0.98      0.98      5572



In [46]:
import pandas as pd
import zipfile
from sklearn.metrics import accuracy_score, classification_report

# Extract the uploaded zip file
with zipfile.ZipFile('yt comments spam dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('yt_dataset')

# Load the CSV file (using the dataset's standard file name inside zip)
# If your zip contains a different CSV name, check the left Files panel
try:
    df_yt = pd.read_csv('yt_dataset/Youtube01-Psy.csv')
except:
    # Fallback to load any CSV found in the extracted directory
    import glob
    csv_file = glob.glob('yt_dataset/*.csv')[0]
    df_yt = pd.read_csv(csv_file)

# Identify comment and label columns
comment_column = 'CONTENT' if 'CONTENT' in df_yt.columns else df_yt.columns[0]
label_column = 'CLASS' if 'CLASS' in df_yt.columns else df_yt.columns[1]

# Preprocess comments using existing pipeline
print("Preprocessing YouTube comments...")
df_yt['cleaned_text'] = df_yt[comment_column].astype(str).apply(preprocess_text)

# Vectorize text feature matrix
X_yt = tfidf.transform(df_yt['cleaned_text']).toarray()
y_yt = df_yt[label_column]

# Generate model predictions
y_yt_pred = model.predict(X_yt)

# Display classification metrics
print("\n=== YouTube Comments Evaluation Results ===")
print(f"Overall Accuracy: {accuracy_score(y_yt, y_yt_pred) * 100:.2f}%\n")
print(classification_report(y_yt, y_yt_pred, target_names=['Clean / Ham', 'Spam']))

Preprocessing YouTube comments...

=== YouTube Comments Evaluation Results ===
Overall Accuracy: 49.95%

              precision    recall  f1-score   support

 Clean / Ham       0.49      0.99      0.66       951
        Spam       0.80      0.03      0.07      1005

    accuracy                           0.50      1956
   macro avg       0.64      0.51      0.36      1956
weighted avg       0.65      0.50      0.35      1956

